In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
sys.path.append('../')
# print(sys.path)
from src.experiment.optuna_tuner import OptunaTuner
from src.utils.experiment_trackers import MLFlowTracker
from typing import Optional
import torch
import lightning.pytorch as pl
# import mlflow
from pathlib import Path
from src.models.lit_model import BaseLitModel
from src.core.params import BaseParams
from src.experiment import StandardRunner
from src.models.model import CrowdCounter
from src.data.datamodule import CrowdDataModule
import mlflow
# Set random seeds for reproducibility
from src import config
from src.utils import helpers
config.set_seed()

print(f"PyTorch: {torch.__version__}")
print(f"Lightning: {pl.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


# def get_datamodule(params: BaseParams):
#     datamodule = CrowdDataModule(params=params)
#     # datamodule.setup(stage='base')
#     return datamodule


# def main():
# import os
# os.environ["TORCH_LOGS"] = "+dynamic"
mlflow.config.enable_async_logging(True)
experiment_name = "crowd_counting"
run_name = None
# study_name = 'check1'
params = BaseParams(
    model_class='MAnet',
    backbone='efficientnet-b2',
    # trainable_backbone=True,
    # decoder_attention_type='scse',
    backbone_weights='imagenet',
    # unfrozen_blocks=('blocks.15', 'blocks.14', 'blocks.13', 'blocks.12', 'blocks.11', 'blocks.10', 'blocks.9', 'blocks.8'),
    crop_size=384,
    batch_size=16,
    decoder_out_channels=32,
    loss_function='bce_mse_ssim',
    ssim_weight=0.6,
    # aug_factor=0.18,
    # num_ops=4,
    epochs=60,
    # l2_reg=0.0009,
    lr=0.00065,
    lr_schedule='clipped_exp',
    # grad_accumulation=16,
    scheduler_kwargs={
        'decay_rate': 0.96,
        'min_lr_pct': 0.01,
    },
)
# architecture = helpers.to_snake_case(params.backbone if params.backbone else params.model_class)
payload, val_results, train_results = StandardRunner(CrowdCounter, MLFlowTracker(experiment_name, run_name), params=params).run()
print(f"\nTraining completed!")
print(f"validation: {val_results}")
print(f"training: {train_results}")
# OptunaTuner(experiment_name,  CrowdCounter, study_name, n_trials=2).run()

# if __name__ == '__main__':
#     main()

/home/jl_fs/workspace/projects/crowd_counting/venv/lib/python3.13/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


PyTorch: 2.11.0+cu130
Lightning: 2.6.1
GPU Available: True
GPU: NVIDIA A30


Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA A30') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


300 images found in ../datasets/ShanghaiTech/part_A/train_data/images
300 h5 files found in ../datasets/ShanghaiTech/part_A/train_data/ground-truth-h5


/home/jl_fs/workspace/projects/crowd_counting/venv/lib/python3.13/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /home/jl_fs/workspace/projects/crowd_counting/notebooks/checkpoints exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/home/jl_fs/workspace/projects/crowd_counting/venv/lib/python3.13/site-packages/lightning/pytorch/utilities/model_summary/model_summary.py:242: Precision bf16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


...

Trainable params: 5.1 M                                                                                            
Non-trainable params: 4.0 M                                                                                        
Total params: 9.1 M                                                                                                
Total estimated model params size (MB): 36                                                                         
Modules in train mode: 411                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/home/jl_fs/workspace/projects/crowd_counting/venv/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/jl_fs/workspace/projects/crowd_counting/venv/lib/python3.13/site-packages/lightning/pytorch/loops/fit_loop.py:317: The number of training batches (15) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
`Trainer.fit` stopped: `max_epochs=60` reached.


🏃 View run run_20260721_022955 at: https://eng-karam-antar-mlflow.duckdns.org/#/experiments/3/runs/8ec433a0d8fa4ce38e01a1a2883d6f26
🧪 View experiment at: https://eng-karam-antar-mlflow.duckdns.org/#/experiments/3


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Comparing Current: 0.2026 vs Historical Best: 0.1350
Model did not outperform historical best. Skipping upload to registry.
📦 Creating archive at /tmp/tmp_0n__kgv/code.zip...
✅ Zip created successfully at /tmp/tmp_0n__kgv/code.zip
🏃 View run run_20260721_022955 at: https://eng-karam-antar-mlflow.duckdns.org/#/experiments/3/runs/8ec433a0d8fa4ce38e01a1a2883d6f26
🧪 View experiment at: https://eng-karam-antar-mlflow.duckdns.org/#/experiments/3

Training completed!
validation: {'val': {'best_val_mae': 111.36954498291016, 'best_val_nae': 0.20260511338710785, 'best_val_rmse': 193.27818298339844, 'best_val_loss': 0.15037456154823303}}
training: {'train': {'best_train_mae': 60.248111724853516, 'best_train_nae': 0.13388727605342865, 'best_train_rmse': 102.16864013671875, 'best_train_epoch_idx': 60.0, 'best_train_loss': 0.13738030195236206}}


In [ ]:
['resnet18', 'resnet34', 'resnet50', 'resnet101', 'resnet152', 'resnext50_32x4d', 'resnext101_32x4d', 'resnext101_32x8d', 'resnext101_32x16d', 'resnext101_32x32d', 'resnext101_32x48d', 'dpn68', 'dpn68b', 'dpn92', 'dpn98', 'dpn107', 'dpn131', 'vgg11', 'vgg11_bn', 'vgg13', 'vgg13_bn', 'vgg16', 'vgg16_bn', 'vgg19', 'vgg19_bn', 'senet154', 'se_resnet50', 'se_resnet101', 'se_resnet152', 'se_resnext50_32x4d', 'se_resnext101_32x4d', 'densenet121', 'densenet169', 'densenet201', 'densenet161', 'inceptionresnetv2', 'inceptionv4', 'efficientnet-b0', 'efficientnet-b1', 'efficientnet-b2', 'efficientnet-b3', 'efficientnet-b4', 'efficientnet-b5', 'efficientnet-b6', 'efficientnet-b7', 'mobilenet_v2', 'xception', 'timm-efficientnet-b0', 'timm-efficientnet-b1', 'timm-efficientnet-b2', 'timm-efficientnet-b3', 'timm-efficientnet-b4', 'timm-efficientnet-b5', 'timm-efficientnet-b6', 'timm-efficientnet-b7', 'timm-efficientnet-b8', 'timm-efficientnet-l2', 'timm-tf_efficientnet_lite0', 'timm-tf_efficientnet_lite1', 'timm-tf_efficientnet_lite2', 'timm-tf_efficientnet_lite3', 'timm-tf_efficientnet_lite4', 'timm-skresnet18', 'timm-skresnet34', 'timm-skresnext50_32x4d', 'mit_b0', 'mit_b1', 'mit_b2', 'mit_b3', 'mit_b4', 'mit_b5', 'mobileone_s0', 'mobileone_s1', 'mobileone_s2', 'mobileone_s3', 'mobileone_s4']

In [10]:
from src.utils.model_registry.mlflow import MLFlowRegistry

payload.force_upload = True
MLFlowRegistry().upload_model(payload)

/home/zeus/miniconda3/envs/cloudspace/lib/python3.13/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.13/site-packages/mlflow/pyfunc/__init__.py:3343: UserWarning: An input example was not provided when logging the model. To ensure the model signature functions correctly, specify the `input_example` parameter. See https://mlflow.org/docs/latest/model/signatures.html#model-input-example for more details about the benefits of using input_example.
  color_warning(


Registered model 'crowd_counting' already exists. Creating a new version of this model...
2026/07/01 17:25:12 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: crowd_counting, version 57
Created version '57' of model 'crowd_counting'.


🏃 View run run_20260701_172010 at: https://eng-karam-antar-mlflow.duckdns.org/#/experiments/3/runs/8d5ba484d9fd47b5a443efb51c6494eb
🧪 View experiment at: https://eng-karam-antar-mlflow.duckdns.org/#/experiments/3


In [ ]:
from src.utils import visualization as vis
model, params = payload.model, payload.params
model.eval()
datamodule = CrowdDataModule(params=params)
datamodule.setup(stage='test')
with torch.no_grad():
    vis.visualize_model_graph(model.model, datamodule=datamodule)

182 images found in /teamspace/lightning_storage/datasets/ShanghaiTech/part_A/test_data/images
182 h5 files found in /teamspace/lightning_storage/datasets/ShanghaiTech/part_A/test_data/ground-truth-h5
************
Feature extraction failed; returning model and environment to normal
*************
'Tensor' object has no attribute 'tl_tensor_label_raw'


In [24]:
import torch
from torchview import draw_graph

dummy_x = torch.randn(1, 3, 256, 256)

# Generates a Graphviz visual of the model
model_graph = draw_graph(model.model, input_size=dummy_x.shape, expand_nested=True, depth=10, show_shapes=True)
model_graph.visual_graph.render(filename='convnext_graph', format='pdf')

'convnext_graph.pdf'

In [34]:
import timm

print(timm.list_models("*vit*"))

['convit_base', 'convit_small', 'convit_tiny', 'crossvit_9_240', 'crossvit_9_dagger_240', 'crossvit_15_240', 'crossvit_15_dagger_240', 'crossvit_15_dagger_408', 'crossvit_18_240', 'crossvit_18_dagger_240', 'crossvit_18_dagger_408', 'crossvit_base_240', 'crossvit_small_240', 'crossvit_tiny_240', 'davit_base', 'davit_base_fl', 'davit_giant', 'davit_huge', 'davit_huge_fl', 'davit_large', 'davit_small', 'davit_tiny', 'efficientvit_b0', 'efficientvit_b1', 'efficientvit_b2', 'efficientvit_b3', 'efficientvit_l1', 'efficientvit_l2', 'efficientvit_l3', 'efficientvit_m0', 'efficientvit_m1', 'efficientvit_m2', 'efficientvit_m3', 'efficientvit_m4', 'efficientvit_m5', 'fastvit_ma36', 'fastvit_mci0', 'fastvit_mci1', 'fastvit_mci2', 'fastvit_mci3', 'fastvit_mci4', 'fastvit_s12', 'fastvit_sa12', 'fastvit_sa24', 'fastvit_sa36', 'fastvit_t8', 'fastvit_t12', 'flexivit_base', 'flexivit_large', 'flexivit_small', 'gcvit_base', 'gcvit_small', 'gcvit_tiny', 'gcvit_xtiny', 'gcvit_xxtiny', 'gemma4_vit_167m', 'g

In [40]:
import torch
import segmentation_models_pytorch as smp
import timm
from src.utils import visualization as vis

# ----------------------------------------------------
# Create SegFormer model with NextViT encoder
# ----------------------------------------------------

model = timm.create_model(
    "levit_128s",
    pretrained=False,
)


model.eval()

# ----------------------------------------------------
# Dummy input
# ----------------------------------------------------

x = torch.randn(1, 3, 512, 512)
vis.visualize_model_graph(model, sample=x)

************
Feature extraction failed; returning model and environment to normal
*************
The size of tensor a (1024) must match the size of tensor b (196) at non-singleton dimension 3
